# 02 · Data Exploration

Les huit requêtes utilisent un seul instantané issu de **01 · Data Preparation**. Chaque résultat complet est exporté en Parquet. Les aperçus sont bornés et le récapitulatif identifie l'instantané et les paramètres employés.

In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
ROOT = next((path for path in (cwd, *cwd.parents) if (path / "src/imdb_loader.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Lancez le notebook depuis le dépôt IMDb ou son dossier notebooks.")
sys.path.insert(0, str(ROOT))

import hashlib
import json
from uuid import uuid4

from pyspark import StorageLevel
from pyspark.sql import functions as F
from src import exploration as query
from src.imdb_loader import data_root, prepared_snapshot, read_prepared, spark_session

DATA = data_root(ROOT)
snapshot = prepared_snapshot(DATA)
validation_year = json.loads((snapshot / "manifest.json").read_text(encoding="utf-8"))["validation_year"]
spark = spark_session("IMDb Data Exploration")
tables = read_prepared(spark, snapshot)
titles, ratings, people = tables["title.basics"], tables["title.ratings"], tables["name.basics"]
crew, principals = tables["title.crew"], tables["title.principals"]
films = query.movies(titles, ratings).persist(StorageLevel.DISK_ONLY)

PARAMETERS = {
    "q01": dict(min_minutes=40, min_genre_films=30, min_z=2.0),
    "q02": dict(first_year=1980, last_year=2025),
    "q03": dict(top_n=3),
    "q04": dict(high_rating=8.0, max_other_mean=6.0, min_other_films=2),
    "q05": dict(min_seasons=3, min_episodes=10, min_votes=1),
    "q06": dict(min_rating=7.0, min_votes=1000),
    "q07": dict(through_year=validation_year, min_years=20, min_films=15, early_years=5, min_votes=100),
    "q08": dict(min_votes=100),
}

results_dir = DATA / "results" / snapshot.name / uuid4().hex
results_dir.mkdir(parents=True)
counts = {}


def save_result(name, frame, order, invalid, *, preview=12, display_columns=None):
    """Materialize each expensive query once, then verify and display its export."""
    path = (results_dir / name).as_posix()
    frame.write.parquet(path)
    result = spark.read.parquet(path)
    assert not result.filter(F.coalesce(invalid, F.lit(True))).take(1), f"Invariant invalide : {name}"
    counts[name] = result.count()
    print(f"{name} : {counts[name]:,} lignes")
    sample = result.orderBy(*order)
    if display_columns is not None:
        sample = sample.select(*display_columns)
    sample.show(preview, truncate=55)
    return result

## Q1 · Durées atypiques par genre

La requête utilise `title.basics` et la vue commune des films. Un long métrage est ici un titre IMDb de type `movie` d'au moins 40 minutes. On conserve les écarts dans les deux sens, à partir de deux écarts-types de population autour de la moyenne du genre et pour au moins 30 films. Un titre peut apparaître dans plusieurs genres.

In [ ]:
q1 = PARAMETERS["q01"]
runtime = query.runtime_outliers(films, **q1)
runtime = save_result("q01_runtime_outliers", runtime,
                      [F.abs("z_score").desc(), "genre", "tconst"],
                      (F.col("runtimeMinutes") < q1["min_minutes"])
                      | (F.col("genre_films") < q1["min_genre_films"])
                      | (F.abs("z_score") < q1["min_z"]))

## Q2 · Trois genres les plus populaires par année

La requête utilise `title.basics`, `title.ratings`. La popularité est la somme des votes IMDb actuels pour les films sortis chaque année entre 1980 et 2025 inclus. Un film contribue à chacun de ses genres. Les films sans note et nombre de votes valides comptent dans le volume de films. Le nom du genre départage les égalités.

In [ ]:
q2, q3 = PARAMETERS["q02"], PARAMETERS["q03"]
ranked = query.genre_rankings(films, **q2).persist(StorageLevel.MEMORY_AND_DISK)
top = save_result("q02_top_genres", ranked.filter(F.col("rank") <= q3["top_n"]),
                  ["year", "rank"], (F.col("rank") < 1) | (F.col("rank") > q3["top_n"])
                  | ~F.col("year").between(q2["first_year"], q2["last_year"])
                  | (F.col("rated_films") > F.col("films")) | (F.col("votes") < 0),
                  display_columns=["year", "rank", "genre", "votes", "films", "rated_films"])
year_sizes = top.groupBy("year").count()
assert not year_sizes.filter(F.col("count") > q3["top_n"]).take(1)
expected_years = spark.range(q2["first_year"], q2["last_year"] + 1).select(F.col("id").alias("year"))
gaps = expected_years.join(year_sizes, "year", "left").filter(
    F.coalesce(F.col("count"), F.lit(0)) < q3["top_n"]
).count()
print(f"Années avec moins de {q3['top_n']} genres : {gaps}")

## Q3 · Genres dominants et autres genres

La requête utilise `title.basics`, `title.ratings`. Pour chaque année, on compare la part des votes, les votes par couple film–genre, le nombre de couples et la note pondérée par les votes. Le détail par genre permet de retrouver le classement et les volumes sous-jacents.

In [ ]:
detail = save_result("q03_genre_detail", ranked, ["year", "rank"],
                     (F.col("films") < F.col("rated_films")) | (F.col("rank") < 1))
comparison = query.compare_genres(ranked, **q3)
comparison = save_result("q03_top_vs_rest", comparison, ["year", F.desc("group")],
                         (F.col("genres") < 1) | (F.col("rated_film_genre_pairs") > F.col("film_genre_pairs"))
                         | ~F.coalesce(F.col("vote_share").between(0.0, 1.0), F.lit(True)))
assert not comparison.groupBy("year").agg(F.sum("vote_share").alias("share")).filter(
    F.abs(F.col("share") - 1.0) > 1e-9
).take(1)
ranked.unpersist()

## Q4 · Performance des réalisateurs

La requête utilise `title.crew`, `title.basics`, `title.ratings`, `name.basics`. Chaque couple réalisateur–film compte une fois, y compris en coréalisation. Il faut exactement un film noté au moins 8 et au moins deux autres films notés, dont la moyenne arithmétique est au plus 6. Les films sans note sont comptés séparément.

In [ ]:
q4 = PARAMETERS["q04"]
directors = save_result("q04_directors", query.directors_performance(films, crew, people, **q4),
                        [F.desc("other_rated_films"), "nconst"],
                        (F.col("high_films") != 1) | (F.col("high_film.averageRating") < q4["high_rating"])
                        | (F.col("other_rated_films") < q4["min_other_films"])
                        | (F.col("other_mean") > q4["max_other_mean"]))

## Q5 · Évolution des séries

La requête utilise `title.episode`, `title.basics`, `title.ratings`. Les séries et miniséries doivent avoir au moins trois saisons numérotées contenant chacune dix épisodes ou plus. La dernière saison est le numéro maximal observé. Les extrêmes utilisent les notes disponibles dans toutes les saisons numérotées. En cas d'égalité, tous les épisodes au minimum doivent être en dernière saison et tous les épisodes au maximum en saison 1 ou 2. La table affiche aussi la couverture des notes.

In [ ]:
q5 = PARAMETERS["q05"]
series = save_result("q05_series", query.series_evolution(titles, tables["title.episode"], ratings, **q5),
                     [F.desc("qualifying_seasons"), "parentTconst"],
                     (F.col("qualifying_seasons") < q5["min_seasons"])
                     | (F.col("worst_episode.seasonNumber") != F.col("last_season"))
                     | (F.col("best_episode.seasonNumber") > 2)
                     | (F.col("rated_episodes") > F.col("episodes")))

## Q6 · Succès dans plusieurs langues

La requête utilise `title.basics`, `title.ratings`, `title.principals`, `name.basics` et un référentiel facultatif de langues originales. Le succès est défini par une note IMDb d'au moins 7 et au moins 1 000 votes. Les crédits principaux acteur et actrice sont dédoublonnés par film et personne.

Les fichiers publics ne donnent aucune langue parlée. Avec `data/reference/original_languages.csv`, on utilise uniquement les langues originales documentées et on conserve une URL de provenance par exemple. Sinon, on calcule un indicateur distinct à partir des langues explicites et concordantes des titres marqués originaux dans `title.akas`. Cet indicateur décrit la langue du titre et sa couverture est mesurée. Il ne peut établir le multilinguisme d'un film ou d'un acteur. Il faut au moins deux films distincts. La section exporte séparément les personnes associées à au moins deux et au moins trois langues.

[Schéma public IMDb](https://data.imdb.com/non-commercial-datasets/) · [Wikidata · langue originale](https://www.wikidata.org/wiki/Property:P364) · [Wikidata · identifiant IMDb](https://www.wikidata.org/wiki/Property:P345)

In [ ]:
actors = query.acting_credits(principals).join(films.select("tconst"), "tconst", "left_semi").persist(
    StorageLevel.DISK_ONLY
)
q6 = PARAMETERS["q06"]
successful = films.filter(
    (F.col("averageRating") >= q6["min_rating"]) & (F.col("numVotes") >= q6["min_votes"])
)
reference_path = DATA / "reference" / "original_languages.csv"
reference_hash = None
if reference_path.is_file():
    language_mode = "verified"
    with reference_path.open("rb") as stream:
        reference_hash = hashlib.file_digest(stream, "sha256").hexdigest()
    language_rows = query.verified_film_languages(spark, reference_path)
else:
    language_mode = "title_proxy"
    language_rows = query.original_title_languages(tables["title.akas"])
    print("Langues parlées indisponibles. L'export porte sur les langues des titres originaux.")

languages = language_rows.join(successful.select("tconst"), "tconst", "left_semi").persist(StorageLevel.DISK_ONLY)
coverage = successful.select("tconst").join(
    languages.select("tconst").distinct().withColumn("covered", F.lit(True)), "tconst", "left"
).agg(F.count("*").alias("successful_films"), F.count("covered").alias("films_with_language")).withColumn(
    "mode", F.lit(language_mode)
).withColumn("coverage", F.try_divide("films_with_language", "successful_films"))
coverage = save_result("q06_language_coverage", coverage, ["mode"],
                       F.col("films_with_language") > F.col("successful_films"))

multi = query.multilingual_actors(successful, actors, people, languages)
multi = save_result(f"q06_{language_mode}_at_least_2", multi, [F.desc("language_count"), "nconst"],
                    (F.col("language_count") < 2) | (F.col("language_count") != F.size("languages")))
multi3 = save_result(f"q06_{language_mode}_at_least_3", multi.filter(F.col("language_count") >= 3),
                     [F.desc("language_count"), "nconst"], F.col("language_count") < 3)
languages.unpersist()

## Q7 · Pic de popularité en début de carrière

La requête utilise `title.basics`, `title.ratings`, `title.principals`, `name.basics`. La carrière observable est l'écart entre le premier et le dernier film datés parmi les crédits principaux. Elle doit couvrir au moins 20 ans et 15 films. Le début couvre les cinq premières années calendaires. Le maximum des notes de ces films avec au moins 100 votes doit dépasser strictement le maximum ultérieur. Une note éligible est nécessaire dans chaque période.

In [ ]:
q7 = PARAMETERS["q07"]
peaks = save_result("q07_early_peaks", query.early_career_peaks(films, actors, people, **q7),
                    [F.desc("early_peak.averageRating"), F.desc("early_peak.numVotes"), "nconst"],
                    (F.col("career_years") < q7["min_years"]) | (F.col("dated_films") < q7["min_films"])
                    | (F.col("early_peak.startYear") >= F.col("first_year") + q7["early_years"])
                    | (F.col("early_peak.averageRating") <= F.col("later_peak_rating")))
actors.unpersist()

## Q8 · Professionnels aux rôles multiples

La requête utilise `title.basics`, `title.ratings`, `title.crew`, `title.principals`, `name.basics`. On combine les crédits principaux et les listes de réalisation et d'écriture. Les catégories actor et actress représentent le même métier. Les apparitions en tant que soi-même et les archives sont exclues. Il faut au moins deux métiers sur l'ensemble de la carrière observable. On compare, pour la même personne, les notes moyennes de ses films à un seul rôle et à plusieurs rôles. Les moyennes couvrent les films avec au moins 100 votes. L'écart reste vide lorsqu'une moyenne est indisponible.

In [ ]:
q8 = PARAMETERS["q08"]
professionals = save_result("q08_professionals",
                            query.multi_role_performance(films, crew, principals, people, **q8),
                            [F.desc("multi_rated_films"), "nconst"],
                            (F.size("career_roles") < 2)
                            | (F.col("multi_rated_films") > F.col("multi_role_films"))
                            | (F.col("single_rated_films") > F.col("single_role_films")),
                            display_columns=["nconst", "primaryName", "career_roles", "multi_role_films",
                                             "single_role_films", "multi_mean_rating", "single_mean_rating",
                                             "rating_delta"])
films.unpersist()

## Index des exports

Chaque sous-dossier contient le DataFrame complet. Le fichier `summary.json` répertorie les volumes et les seuils lorsque tous les exports et contrôles ont abouti.

In [ ]:
summary = {
    "prepared_snapshot": snapshot.name,
    "parameters": PARAMETERS,
    "q06_language_mode": language_mode,
    "q06_reference_sha256": reference_hash,
    "q06_coverage": coverage.first().asDict(),
    "q02_years_with_fewer_than_top_n": gaps,
    "rows": counts,
}
(results_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(results_dir)
print(json.dumps(counts, indent=2))
spark.stop()